In [1]:
# Cellule 1 - Charger les bibliothèques
from pathlib import Path
import json
import shutil
import joblib
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
# Cellule 2 - Charger les données
dataset = fetch_california_housing(as_frame=True)
df = dataset.frame

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(df.shape)
df.head()


(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
# Cellule 2b — Configurer MLFlow
mlflow.set_tracking_uri((Path("..") / "mlruns").resolve().as_uri())
mlflow.set_experiment("housing_ab_models")
print("Expérience MLFlow configurée : housing_ab_models")

2026/05/12 15:42:06 INFO mlflow.tracking.fluent: Experiment with name 'housing_ab_models' does not exist. Creating a new experiment.


Expérience MLFlow configurée : housing_ab_models


In [4]:
# Cellule 3 - Fonction d’évaluation

def evaluate_pipeline(pipeline, X_test, y_test):
    preds = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }


In [5]:
# Cellule 4 - Entraîner le modèle A (champion) avec MLFlow
with mlflow.start_run(run_name="model_A_RandomForest"):

    params_a = {
        "model_type":   "RandomForestRegressor",
        "n_estimators": 30,
        "max_depth":    8,
        "random_state": 42,
        "role":         "champion",
    }
    mlflow.log_params(params_a)

    model_a = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  RandomForestRegressor(
            n_estimators=params_a["n_estimators"],
            max_depth=params_a["max_depth"],
            random_state=params_a["random_state"],
            n_jobs=-1,
        )),
    ])
    model_a.fit(X_train, y_train)

    metrics_a = evaluate_pipeline(model_a, X_test, y_test)
    mlflow.log_metrics(metrics_a)

    mlflow.sklearn.log_model(
        model_a,
        artifact_path="model",
        registered_model_name="housing_model_A",
    )
    print("Modèle A :", metrics_a)

2026/05/12 15:42:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Modèle A : {'mae': 0.4060793276787333, 'rmse': 0.5879996510519797, 'r2': 0.7361559670467321}


Successfully registered model 'housing_model_A'.
Created version '1' of model 'housing_model_A'.


In [6]:
# Cellule 5 - Entraîner le modèle B (challenger) avec MLFlow
with mlflow.start_run(run_name="model_B_GradientBoosting"):

    params_b = {
        "model_type":    "GradientBoostingRegressor",
        "n_estimators":  80,
        "learning_rate": 0.05,
        "random_state":  42,
        "role":          "challenger",
    }
    mlflow.log_params(params_b)

    model_b = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  GradientBoostingRegressor(
            n_estimators=params_b["n_estimators"],
            learning_rate=params_b["learning_rate"],
            random_state=params_b["random_state"],
        )),
    ])
    model_b.fit(X_train, y_train)

    metrics_b = evaluate_pipeline(model_b, X_test, y_test)
    mlflow.log_metrics(metrics_b)

    mlflow.sklearn.log_model(
        model_b,
        artifact_path="model",
        registered_model_name="housing_model_B",
    )
    print("Modèle B :", metrics_b)

2026/05/12 15:42:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Successfully registered model 'housing_model_B'.


Modèle B : {'mae': 0.4256443235369947, 'rmse': 0.6013137122748726, 'r2': 0.7240722655493608}


Created version '1' of model 'housing_model_B'.


In [7]:
# Cellule 6 - Sauvegarder modèles et métriques
models_dir = Path("../artifacts/models")
metrics_dir = Path("../artifacts/metrics")
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model_a, models_dir / "model_v1.joblib")
joblib.dump(model_b, models_dir / "model_v2.joblib")

with open(metrics_dir / "metrics_v1.json", "w", encoding="utf-8") as f:
    json.dump(metrics_a, f, indent=2)

with open(metrics_dir / "metrics_v2.json", "w", encoding="utf-8") as f:
    json.dump(metrics_b, f, indent=2)

shutil.copy(models_dir / "model_v1.joblib", models_dir / "model_champion.joblib")
shutil.copy(models_dir / "model_v2.joblib", models_dir / "model_challenger.joblib")

print("Modèles A et B sauvegardés")


Modèles A et B sauvegardés
